In [0]:
%python
# ===== Cell 1: where to log =====
import mlflow
 
user = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/bug_triage_eval")
prompts = {r.version: r.team_guide for r in spark.table("support.bugzilla.prompts").collect()}
 
# Settings every run used. They are not stored in eval_runs, because they were the
# same for every run (they live in the ai_query call), so we record them here.
# If you ever change one in the harness, change it here too.
CALL_SETTINGS = {"max_tokens": 2000, "temperature": 0.0, "description_chars": 2000,
                 "endpoint_type": "pay-per-token (ai_query)"}

In [0]:
%python
routing = spark.sql("""
  SELECT run_at, model, prompt_version, eval_table,
         COUNT(*) AS bugs,
         AVG(CASE WHEN predicted = true_team THEN 1 ELSE 0 END) AS accuracy,
         COUNT_IF(predicted IS NULL OR predicted = '') AS blanks
  FROM support.bugzilla.eval_runs
  GROUP BY run_at, model, prompt_version, eval_table
""").collect()
 
cost = {(r.model, r.prompt_version): r.cost_per_1k for r in spark.sql("""
  SELECT tu.model, prompt_version,
         1000 * (AVG(prompt_tokens) / 1e6 * MAX(input_per_m)
               + AVG(completion_tokens) / 1e6 * MAX(output_per_m)) AS cost_per_1k
  FROM support.bugzilla.token_usage tu
  JOIN support.bugzilla.model_prices mp ON tu.model = mp.model
  GROUP BY tu.model, prompt_version
""").collect()}
 
for r in routing:
    with mlflow.start_run(run_name=f"routing | {r.model} | {r.prompt_version} | {r.eval_table.split('.')[-1]}"):
        mlflow.set_tag("task", "routing")
        mlflow.set_tag("run_at", str(r.run_at))
        mlflow.log_params({"model": r.model, "prompt_version": r.prompt_version,
                           "eval_table": r.eval_table, "bugs": r.bugs, **CALL_SETTINGS})
        mlflow.log_metrics({"accuracy": r.accuracy, "blanks": r.blanks})
        if (r.model, r.prompt_version) in cost:
            mlflow.log_metric("cost_per_1k_usd", cost[(r.model, r.prompt_version)])
        mlflow.log_text(prompts.get(r.prompt_version, ""), "prompt.txt")
print(len(routing), "routing runs logged")

In [0]:
%python
# ===== Cell 3: urgency runs =====
# startswith() so an answer like "not urgent. The bug is..." still counts.
urgency = spark.sql("""
  SELECT run_at, model, prompt_version,
         COUNT_IF(startswith(predicted, 'urgent') AND is_urgent) / COUNT_IF(is_urgent) AS recall,
         COUNT_IF(startswith(predicted, 'urgent') AND NOT is_urgent) / COUNT_IF(NOT is_urgent) AS false_alarm_rate,
         COUNT_IF(predicted NOT IN ('urgent', 'not urgent')) AS off_format
  FROM support.bugzilla.urgency_runs
  GROUP BY run_at, model, prompt_version
""").collect()
 
for r in urgency:
    with mlflow.start_run(run_name=f"urgency | {r.model} | {r.prompt_version}"):
        mlflow.set_tag("task", "urgency")
        mlflow.set_tag("run_at", str(r.run_at))
        mlflow.log_params({"model": r.model, "prompt_version": r.prompt_version,
                           "eval_table": "support.bugzilla.urgency_set",
                           "team_shown": r.prompt_version != "u1", **CALL_SETTINGS})
        mlflow.log_metrics({"recall": r.recall, "false_alarm_rate": r.false_alarm_rate,
                            "off_format": r.off_format})
        mlflow.log_text(prompts.get(r.prompt_version, ""), "prompt.txt")
print(len(urgency), "urgency runs logged")